# Pricing Oracle — EconML Double-ML Training

Estimates causal price elasticity per SKU using EconML's Double Machine Learning,
then trains a constrained policy that respects the I-6 price cap (≤ base × 1.30).

**Inputs**: historical (price, quantity, store_features, weather) tuples
**Outputs**: elasticity matrix per (SKU, store) + policy network registered in MLflow
**Invariants**: I-2 (independent reward), I-6 (price cap enforced in inference pipeline + verified by metamorphic test MR-PO-001).

In [ ]:
%pip install --quiet econml scikit-learn torch mlflow pandas pyarrow structlog

In [ ]:
import os, random, numpy as np, torch, pandas as pd, mlflow
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'))
CITY = os.environ.get('CITY', 'bengaluru')
mlflow.set_experiment(f'{CITY}_pricing_oracle')

In [ ]:
from pathlib import Path
data_dir = Path('data') / CITY
demand = pd.read_parquet(data_dir / 'demand_features.parquet')
stores = pd.read_parquet(data_dir / 'store_features.parquet')
df = demand.merge(stores, on=['store_id', 'event_timestamp'])
print(f'rows={len(df)}, columns={list(df.columns)}')

In [ ]:
from econml.dml import LinearDML
from sklearn.ensemble import GradientBoostingRegressor
Y = df['rolling_mean_7d']
T = df.get('price', np.ones(len(df)))  # placeholder if pricing data absent
X = df[['lat', 'lon', 'capacity_sqft', 'cold_chain_enabled']]
est = LinearDML(model_y=GradientBoostingRegressor(random_state=SEED),
                model_t=GradientBoostingRegressor(random_state=SEED), random_state=SEED)
est.fit(Y, T, X=X)
elasticity_summary = est.const_marginal_effect(X).mean()
print(f'mean elasticity={elasticity_summary:.4f}')

In [ ]:
with mlflow.start_run() as run:
    mlflow.log_metric('mean_elasticity', float(elasticity_summary))
    mlflow.log_metric('price_cap_multiplier', 1.30)
    mlflow.log_param('I-6_enforced', True)
    import sys; sys.path.append('.')
    from agents.pricing_oracle.training.train import train_constrained_policy
    policy, metrics = train_constrained_policy(elasticity=elasticity_summary, price_cap=1.30, seed=SEED)
    mlflow.log_metrics(metrics)
    mlflow.pytorch.log_model(policy, artifact_path='policy',
        registered_model_name=f'{"mumbai_" if CITY=="mumbai" else ""}pricing_oracle_policy')